# Broken vs Corrected Backtest

## The problem in one sentence
A backtest using raw (unadjusted) prices generates wrong signals on corporate action dates —
and the wrong signals systematically favour the wrong side of the trade.

## What this notebook demonstrates

We run a simple 12-month price-momentum strategy on TATASTEEL, which had a bonus issue
and a subsequent split in our sample window.

- **Broken backtest** — raw prices, no adjustment
- **Corrected backtest** — TickerTruth-adjusted prices

The bonus issue drops the raw price by ~83% on a single day. A lookback window
that straddles that date sees a massive negative 12-month return and issues a *short* signal —
the exact opposite of the correct signal.

**Data source:** `sample_data/adjustment_factors.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

_cwd = Path.cwd()
SAMPLE_DATA = (
    _cwd / "sample_data"
    if (_cwd / "sample_data").exists()
    else _cwd.parent / "sample_data"
)

adj = pd.read_parquet(SAMPLE_DATA / "adjustment_factors.parquet")
adj["ex_date"] = pd.to_datetime(adj["ex_date"])

# TATASTEEL events only
tata_events = adj[adj["symbol"] == "TATASTEEL"].sort_values("ex_date").copy()
print("TATASTEEL corporate action events:")
print(
    tata_events[
        ["ex_date", "action_type", "adjustment_factor", "cumulative_adjustment_factor"]
    ].to_string(index=False)
)

## Build synthetic raw price series for TATASTEEL

In [ ]:
rng = np.random.default_rng(7)

dates = pd.date_range("2016-01-01", "2024-12-31", freq="B")
n = len(dates)

# True underlying value: steady grower
true_log_returns = rng.normal(0.00025, 0.018, size=n)
true_prices = 400 * np.exp(np.cumsum(true_log_returns))

# Apply TATASTEEL events to produce raw series
raw = true_prices.copy()
for _, ev in tata_events.iterrows():
    mask = dates >= ev["ex_date"]
    raw[mask] *= ev["adjustment_factor"]


# Adjusted series: divide by cumulative factor
def cum_factor_at(date, events):
    past = events[events["ex_date"] <= date]
    return past.iloc[-1]["cumulative_adjustment_factor"] if not past.empty else 1.0


cum = np.array([cum_factor_at(d, tata_events) for d in dates])
adjusted = raw / cum

df = pd.DataFrame({"raw": raw, "adjusted": adjusted}, index=dates)
print(f"Raw price range:      INR {df['raw'].min():.0f} – {df['raw'].max():.0f}")
print(
    f"Adjusted price range: INR {df['adjusted'].min():.0f} – {df['adjusted'].max():.0f}"
)

## Run the momentum strategy on both series

Strategy: at the start of each month, compute the 12-month price return.
- If return > 0 → **long** (buy)
- If return < 0 → **short** (sell / avoid)

Hold for one month; rebalance monthly.

In [ ]:
def run_momentum(prices, lookback_months=12):
    """Monthly 12-month momentum strategy. Returns a DataFrame of signals and returns."""
    monthly = prices.resample("ME").last()
    ret_12m = monthly.pct_change(lookback_months)  # 12-month lookback return
    signal = np.sign(ret_12m)  # +1 long, -1 short
    fwd_ret = monthly.pct_change(1).shift(-1)  # next month actual return
    strat = signal * fwd_ret  # strategy return
    return pd.DataFrame(
        {
            "price": monthly,
            "momentum_12m": ret_12m,
            "signal": signal,
            "strategy_return": strat,
        }
    )


broken = run_momentum(df["raw"])
corrected = run_momentum(df["adjusted"])

# Focus on 2017 onwards (first 12 months used as lookback seed)
broken = broken.loc["2017":]
corrected = corrected.loc["2017":]

print("Broken (raw) strategy — signal sample around TATASTEEL bonus event (2018-02):")
bad_window = broken.loc["2018-01":"2018-06"]
print(bad_window[["price", "momentum_12m", "signal"]].round(3).to_string())

In [ ]:
print("Corrected (adjusted) strategy — same window:")
print(
    corrected.loc["2018-01":"2018-06"][["price", "momentum_12m", "signal"]]
    .round(3)
    .to_string()
)
print()
print(
    "The broken strategy issues a SHORT signal in Feb 2018 because raw 12m return is -83%."
)
print("The corrected strategy correctly sees positive momentum and stays LONG.")

## Cumulative P&L comparison

In [ ]:
broken_cum = (1 + broken["strategy_return"].fillna(0)).cumprod()
corrected_cum = (1 + corrected["strategy_return"].fillna(0)).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

# Top panel: raw vs adjusted price
ax = axes[0]
ax.plot(df.index, df["raw"], color="#e05252", lw=0.8, label="Raw price")
ax.plot(df.index, df["adjusted"], color="#3b82f6", lw=0.8, label="Adjusted price")
for _, ev in tata_events.iterrows():
    ax.axvline(ev["ex_date"], color="#888", ls="--", lw=0.8)
    ax.text(
        ev["ex_date"],
        df["raw"].max() * 0.85,
        f" {ev['action_type']}",
        fontsize=7,
        color="#555",
    )
ax.set_ylabel("Price (INR)")
ax.set_title("TATASTEEL — Raw vs Adjusted Price", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bottom panel: cumulative P&L
ax2 = axes[1]
ax2.plot(
    broken_cum.index,
    broken_cum.values,
    color="#e05252",
    lw=1.2,
    label="Broken backtest (raw prices)",
)
ax2.plot(
    corrected_cum.index,
    corrected_cum.values,
    color="#10b981",
    lw=1.2,
    label="Corrected backtest (adjusted prices)",
)
ax2.axhline(1.0, color="#888", lw=0.6, ls=":")

# Highlight the wrong-signal period
ax2.axvspan(
    pd.Timestamp("2018-02-01"),
    pd.Timestamp("2018-08-01"),
    alpha=0.12,
    color="red",
    label="Wrong signal period",
)

for _, ev in tata_events.iterrows():
    ax2.axvline(ev["ex_date"], color="#888", ls="--", lw=0.8)

ax2.set_ylabel("Cumulative return (1 = start)")
ax2.set_title("Cumulative P&L — Momentum Strategy on TATASTEEL", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Summary statistics
for label, strat in [("Broken (raw)", broken), ("Corrected (adjusted)", corrected)]:
    rets = strat["strategy_return"].dropna()
    ann_ret = (1 + rets.mean()) ** 12 - 1
    ann_vol = rets.std() * (12**0.5)
    sharpe = ann_ret / ann_vol if ann_vol else 0
    wrong = (rets < -0.05).sum()  # months with >5% loss
    print(f"{label}:")
    print(f"  Annualised return: {ann_ret:+.1%}")
    print(f"  Annualised vol:    {ann_vol:.1%}")
    print(f"  Sharpe ratio:      {sharpe:.2f}")
    print(f"  Months with >5%% loss: {wrong}")
    print()

---
## Key takeaway

The broken backtest does not just under-perform — it gives **the wrong signal at the wrong time**.
The short signal issued after the bonus ex-date would cause a real portfolio to:
1. Exit a winning long position right before recovery
2. Pay transaction costs twice (exit + re-entry)
3. Potentially miss months of post-event upside

This is not a corner case. Any momentum, mean-reversion, or ML model that ingests
raw NSE prices without adjustment factors will reproduce this error for *every* split
and bonus in its training window — silently, with no error message.

**TickerTruth adjustment factors eliminate this class of error entirely.**